# Bayesian Spatial Panel Models

This notebook demonstrates the panel-model classes in `bayespecon`:
- Fixed effects / pooled panel models: `OLSPanelFE`, `SARPanelFE`, `SEMPanelFE`, `SDMPanelFE`, `SDEMPanelFE`
- Random effects panel models: `OLSPanelRE`, `SARPanelRE`, `SEMPanelRE`

It also demonstrates Bayesian model comparison via WAIC, LOO-CV, and Bayes factors.

## Panel Model Equations

Let observations be stacked by time, then unit (panel_g convention).

Fixed-effects / pooled classes:
- OLS panel: $y_{it} = x_{it}'\beta + a_i + \tau_t + \epsilon_{it}$
- SAR panel: $y_{it} = \rho (Wy)_{it} + x_{it}'\beta + a_i + \tau_t + \epsilon_{it}$
- SEM panel: $y_{it} = x_{it}'\beta + a_i + \tau_t + u_{it},\; u_{it}=\lambda (Wu)_{it}+\epsilon_{it}$
- SDM panel: $y_{it} = \rho (Wy)_{it} + x_{it}'\beta + (Wx)_{it}'\theta + a_i + \tau_t + \epsilon_{it}$
- SDEM panel: $y_{it} = x_{it}'\beta + (Wx)_{it}'\theta + a_i + \tau_t + u_{it},\; u_{it}=\lambda (Wu)_{it}+\epsilon_{it}$

Random-effects classes:
- OLS random effects: $y_{it} = x_{it}'\beta + \alpha_i + \epsilon_{it},\; \alpha_i \sim N(0, \sigma_\alpha^2)$
- SAR random effects: $y_{it} = \rho (Wy)_{it} + x_{it}'\beta + \alpha_i + \epsilon_{it},\; \alpha_i \sim N(0, \sigma_\alpha^2)$
- SEM random effects: $y_{it} = x_{it}'\beta + \alpha_i + u_{it},\; u_{it}=\lambda (Wu)_{it}+\epsilon_{it},\; \alpha_i \sim N(0, \sigma_\alpha^2)$

The `model` argument selects the pooled or fixed-effects specification for the FE classes:
- `0` pooled
- `1` unit FE
- `2` time FE
- `3` two-way FE

The RE classes are hierarchical models with unit-level random intercepts, so they internally use the pooled design and estimate `sigma_alpha` directly.

In [ ]:
import arviz as az
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from libpysal.graph import Graph

from bayespecon import (
    OLSPanelFE,
    OLSPanelRE,
    SARPanelFE,
    SARPanelRE,
    SDEMPanelFE,
    SDMPanelFE,
    SEMPanelFE,
    SEMPanelRE,
    dgp,
)

az.style.use("arviz-white")

## Build a Balanced Panel Dataset

For pedagogy, we generate synthetic panel data from the `bayespecon.dgp` module on a regular polygon grid and then assemble a formula-ready DataFrame.


In [ ]:
# Generate base geometry via cross-sectional DGP, then simulate panel data via panel DGP.
rng = np.random.default_rng(2026)
xcols = ["poverty", "rev_rating", "num_spots", "crowded"]
ycol = "price_pp"

base_gdf = dgp.simulate_sar(n=8, seed=2026, create_gdf=True, geometry_type="polygon")
W = Graph.build_contiguity(base_gdf, rook=False).transform("r")
N = len(base_gdf)
T = 4
beta = np.array([1.5, -0.8, 0.6, 0.4, -0.5], dtype=float)

panel_sim = dgp.simulate_panel_sar_fe(
    N=N,
    T=T,
    rho=0.35,
    beta=beta,
    sigma=1.0,
    sigma_alpha=0.5,
    rng=rng,
    W=W,
)

panel = pd.DataFrame(
    {
        "unit": panel_sim["unit"],
        "time": panel_sim["time"],
        ycol: panel_sim["y"],
    }
)
for j, name in enumerate(xcols, start=1):
    panel[name] = panel_sim["X"][:, j]

## Shared Helpers

In [ ]:
def fit_panel_model(
    model_cls, formula, data, W, model=3, draws=300, tune=300, chains=2, seed=42
):
    """Fit a panel model and return (model, summary, effects_df).

    Uses minimal MCMC settings for pedagogical demonstration.
    Increase draws/tune for real analyses.
    """
    m = model_cls(
        formula=formula,
        data=data,
        W=W,
        unit_col="unit",
        time_col="time",
        model=model,
        logdet_method="eigenvalue",
    )
    m.fit(
        draws=draws,
        tune=tune,
        chains=chains,
        target_accept=0.9,
        random_seed=seed,
        progressbar=False,
        idata_kwargs={"log_likelihood": True},
    )
    summary = m.summary(round_to=3)
    effects_df = pd.DataFrame(m.spatial_effects())
    return m, summary, effects_df


def diagnostics_table(idata, var_names):
    """Show key MCMC diagnostics for the given parameters."""
    cols = ["mean", "sd", "ess_bulk", "ess_tail", "r_hat"]
    return az.summary(idata, var_names=var_names, round_to=3)[cols]


def show_trace(idata, var_names, title):
    """Plot trace plots for the given parameters."""
    az.plot_trace(idata, var_names=var_names)
    plt.suptitle(title, y=1.02)
    plt.tight_layout()
    plt.show()

## Fit: OLSPanelFE

In [ ]:
formula = "price_pp ~ poverty + rev_rating + num_spots + crowded"
ols_panel, summary_ols, effects_ols = fit_panel_model(
    OLSPanelFE, formula, panel, W, model=3
)
display(summary_ols)
display(effects_ols)
display(diagnostics_table(ols_panel.inference_data, ["beta", "sigma"]))
show_trace(ols_panel.inference_data, ["sigma"], "OLSPanelFE trace")

## Fit: SARPanelFE

In [ ]:
sar_panel, summary_sar, effects_sar = fit_panel_model(
    SARPanelFE, formula, panel, W, model=3
)
display(summary_sar)
display(effects_sar)
display(diagnostics_table(sar_panel.inference_data, ["rho", "beta", "sigma"]))
show_trace(sar_panel.inference_data, ["rho", "sigma"], "SARPanelFE trace")

## Fit: SEMPanelFE

In [ ]:
sem_panel, summary_sem, effects_sem = fit_panel_model(
    SEMPanelFE, formula, panel, W, model=3
)
display(summary_sem)
display(effects_sem)
display(diagnostics_table(sem_panel.inference_data, ["lam", "beta", "sigma"]))
show_trace(sem_panel.inference_data, ["lam", "sigma"], "SEMPanelFE trace")

## Fit: SDMPanelFE

In [ ]:
sdm_panel, summary_sdm, effects_sdm = fit_panel_model(
    SDMPanelFE, formula, panel, W, model=3
)
display(summary_sdm)
display(effects_sdm)
display(diagnostics_table(sdm_panel.inference_data, ["rho", "beta", "sigma"]))
show_trace(sdm_panel.inference_data, ["rho", "sigma"], "SDMPanelFE trace")

## Fit: SDEMPanelFE

In [ ]:
sdem_panel, summary_sdem, effects_sdem = fit_panel_model(
    SDEMPanelFE, formula, panel, W, model=3
)
display(summary_sdem)
display(effects_sdem)
display(diagnostics_table(sdem_panel.inference_data, ["lam", "beta", "sigma"]))
show_trace(sdem_panel.inference_data, ["lam", "sigma"], "SDEMPanelFE trace")

### MCMC adequacy for the spatial parameter

Panel SAR/SDEM models inherit the slow-mixing behaviour of $\rho$
(or $\lambda$) documented by Wolf, Anselin & Arribas-Bel (2018)
{cite:p}`wolf2018StochasticEfficiency`. The
`spatial_mcmc_diagnostic` helper checks ESS, sampler yield, $\hat{R}$,
and HPDI stability for the spatial scalar.


In [ ]:
from bayespecon import spatial_mcmc_diagnostic

spatial_mcmc_diagnostic(sdm_panel, emit_warnings=False).to_frame()

## Random-Effects Panel Models

The random-effects models replace deterministic unit fixed effects with latent unit intercepts $\alpha_i \sim N(0, \sigma_\alpha^2)$. That lets the notebook estimate the variance of unit heterogeneity directly and compare FE and RE behavior on the same synthetic balanced panel.

### Fit: OLSPanelRE

In [ ]:
ols_panel_re, summary_ols_re, effects_ols_re = fit_panel_model(
    OLSPanelRE, formula, panel, W, model=0
)
display(summary_ols_re)
display(effects_ols_re)
display(
    diagnostics_table(ols_panel_re.inference_data, ["beta", "sigma", "sigma_alpha"])
)
show_trace(ols_panel_re.inference_data, ["sigma", "sigma_alpha"], "OLSPanelRE trace")

### Fit: SARPanelRE

In [ ]:
sar_panel_re, summary_sar_re, effects_sar_re = fit_panel_model(
    SARPanelRE, formula, panel, W, model=0
)
display(summary_sar_re)
display(effects_sar_re)
display(
    diagnostics_table(
        sar_panel_re.inference_data, ["rho", "beta", "sigma", "sigma_alpha"]
    )
)
show_trace(
    sar_panel_re.inference_data, ["rho", "sigma", "sigma_alpha"], "SARPanelRE trace"
)

### Fit: SEMPanelRE

In [ ]:
sem_panel_re, summary_sem_re, effects_sem_re = fit_panel_model(
    SEMPanelRE, formula, panel, W, model=0
)
display(summary_sem_re)
display(effects_sem_re)
display(
    diagnostics_table(
        sem_panel_re.inference_data, ["lam", "beta", "sigma", "sigma_alpha"]
    )
)
show_trace(
    sem_panel_re.inference_data, ["lam", "sigma", "sigma_alpha"], "SEMPanelRE trace"
)

## Model Comparison (WAIC and LOO) Across FE and RE Models

In [ ]:
# Collect all fitted panel models for comparison
panel_models = {
    "OLSPanelFE": ols_panel,
    "SARPanelFE": sar_panel,
    "SEMPanelFE": sem_panel,
    "SDMPanelFE": sdm_panel,
    "SDEMPanelFE": sdem_panel,
    "OLSPanelRE": ols_panel_re,
    "SARPanelRE": sar_panel_re,
    "SEMPanelRE": sem_panel_re,
}
idata_dict = {name: m.inference_data for name, m in panel_models.items()}

# WAIC and LOO comparison
for ic in ("waic", "loo"):
    try:
        cmp = az.compare(idata_dict, ic=ic, method="BB-pseudo-BMA")
        print(f"\n{ic.upper()} comparison")
        display(cmp)
    except Exception as e:
        print(f"{ic.upper()} comparison not available: {type(e).__name__}: {e}")

# Per-model IC table
rows = []
for name, model in panel_models.items():
    idata = model.inference_data
    row = {"model": name}
    try:
        waic_res = az.waic(idata)
        row["elpd_waic"] = float(waic_res.elpd_waic)
        row["p_waic"] = float(waic_res.p_waic)
    except Exception:
        row["elpd_waic"] = np.nan
        row["p_waic"] = np.nan
    try:
        loo_res = az.loo(idata)
        row["elpd_loo"] = float(loo_res.elpd_loo)
        row["p_loo"] = float(loo_res.p_loo)
    except Exception:
        row["elpd_loo"] = np.nan
        row["p_loo"] = np.nan
    rows.append(row)

pd.DataFrame(rows).sort_values("model").reset_index(drop=True)

## Notes

- This notebook uses modest draw counts for speed. For real analyses, increase `draws` and `tune`.
- If you see divergences or tree-depth warnings, increase `target_accept` (for example 0.95-0.99).
- For production inference, validate sensitivity to FE specification (`model=0/1/2/3`) and priors.

## Bayesian Panel LM Diagnostics

The `bayespecon.diagnostics` module provides a full suite of Bayesian Lagrange Multiplier (LM) tests for panel models, following the framework of Dogan et al. (2021) with panel-specific formulas from Anselin (2008), Elhorst (2014), and Koley & Bera (2024).

These tests help answer the question: *which spatial panel specification is most appropriate?* They test for omitted spatial lag ($\rho$), spatial error ($\lambda$), and spatially lagged covariates ($\gamma$), both individually and jointly, with robust variants that account for the presence of other spatial effects.

The decision tree mirrors the classical Anselin (1988) / Koley-Bera (2024) approach, but uses the full posterior distribution rather than point estimates — yielding Bayesian p-values and credible intervals for each LM statistic.

In [ ]:
from bayespecon.diagnostics.bayesian_lmtests import (
    bayesian_panel_lm_error_test,
    bayesian_panel_lm_lag_test,
    bayesian_panel_lm_sdm_joint_test,
    bayesian_panel_lm_slx_error_joint_test,
    bayesian_panel_lm_wx_test,
    bayesian_panel_robust_lm_error_sdem_test,
    bayesian_panel_robust_lm_error_test,
    bayesian_panel_robust_lm_lag_sdm_test,
    bayesian_panel_robust_lm_lag_test,
    bayesian_panel_robust_lm_wx_test,
)

### Standard Panel LM Tests (from OLS null)

The four standard panel LM tests use the OLS panel model as the null. They test for spatial lag, spatial error, and joint SDM/SDEM alternatives.

In [ ]:
# Standard panel LM tests (from OLS null)
panel_standard = pd.DataFrame(
    [
        bayesian_panel_lm_lag_test(ols_panel).to_series(),
        bayesian_panel_lm_error_test(ols_panel).to_series(),
        bayesian_panel_lm_sdm_joint_test(ols_panel).to_series(),
        bayesian_panel_lm_slx_error_joint_test(ols_panel).to_series(),
    ],
    index=["LM-Lag", "LM-Error", "LM-SDM Joint", "LM-SLX-Error Joint"],
)

panel_standard

### Robust Panel LM Tests (from OLS null)

The robust variants (Elhorst, 2014) test one spatial effect while accounting for the possible local presence of the other. For example, the robust LM-Lag tests for $\rho$ robust to $\lambda$, and vice versa.

In [ ]:
# Robust panel LM tests (from OLS null)
panel_robust = pd.DataFrame(
    [
        bayesian_panel_robust_lm_lag_test(ols_panel).to_series(),
        bayesian_panel_robust_lm_error_test(ols_panel).to_series(),
    ],
    index=["Robust LM-Lag", "Robust LM-Error"],
)

panel_robust

### SDM/SDEM Variant Tests

These tests address the Koley & Bera (2024) decision tree for choosing between SAR/SEM and SDM/SDEM specifications. They require fitting intermediate models (SAR or SLX) as the alternative model.

- **LM-WX** (from SAR null): Should we extend SAR → SDM by adding $\gamma$?
- **Robust LM-Lag-SDM** (from SLX null): Should we extend SLX → SDM by adding $\rho$, robust to $\gamma$?
- **Robust LM-WX** (from SAR null): Should we extend SAR → SDM by adding $\gamma$, robust to $\rho$?
- **Robust LM-Error-SDEM** (from SLX null): Should we extend SLX → SDEM by adding $\lambda$, robust to $\gamma$?

We need to fit an SLX panel model first to serve as the null for the robust SDM/SDEM tests.

In [ ]:
from bayespecon import SLXPanelFE

slx_panel, summary_slx, effects_slx = fit_panel_model(
    SLXPanelFE, formula, panel, W, model=3
)
display(summary_slx)

In [ ]:
# SDM/SDEM variant panel LM tests
panel_sdem = pd.DataFrame(
    [
        bayesian_panel_lm_wx_test(sar_panel).to_series(),
        bayesian_panel_robust_lm_lag_sdm_test(slx_panel).to_series(),
        bayesian_panel_robust_lm_wx_test(sar_panel).to_series(),
        bayesian_panel_robust_lm_error_sdem_test(slx_panel).to_series(),
    ],
    index=["LM-WX", "Robust LM-Lag-SDM", "Robust LM-WX", "Robust LM-Error-SDEM"],
)

panel_sdem

### Interpreting the Panel LM Decision Tree

The panel LM tests follow the same decision logic as the classical Anselin/Koley-Bera approach, but with Bayesian p-values:

| Scenario | Significant? | Interpretation |
|---|---|---|
| LM-Lag ✓, LM-Error ✗ | Lag only | Choose SAR panel |
| LM-Lag ✗, LM-Error ✓ | Error only | Choose SEM panel |
| Both significant → check robust tests | | |
| Robust LM-Lag ✓, Robust LM-Error ✗ | Lag dominates | Choose SAR panel |
| Robust LM-Lag ✗, Robust LM-Error ✓ | Error dominates | Choose SEM panel |
| Both robust significant | Both effects | Consider SDM or SDEM |
| Joint LM-SDM ✓ | SDM preferred over OLS | Fit SDM panel |
| Joint LM-SLX-Error ✓ | SDEM preferred over OLS | Fit SDEM panel |
| LM-WX ✓ (from SAR) | WX effects present | Extend SAR → SDM |
| Robust LM-Lag-SDM ✓ (from SLX) | Lag needed beyond WX | Extend SLX → SDM |
| Robust LM-WX ✓ (from SAR) | WX needed beyond lag | Extend SAR → SDM |
| Robust LM-Error-SDEM ✓ (from SLX) | Error needed beyond WX | Extend SLX → SDEM |

A Bayesian p-value below 0.05 (or a threshold you choose) indicates evidence against the null hypothesis. The `credible_interval` field shows the posterior uncertainty in the LM statistic itself.

In [ ]:
from scipy import stats as sp_stats

# All panel LM tests in a single table
all_panel = pd.concat([panel_standard, panel_robust, panel_sdem])

# Add chi-squared reference values
all_panel["chi2_ref_95"] = all_panel["df"].apply(lambda df: sp_stats.chi2.ppf(0.95, df))

all_panel

- LM-Lag is highly significant (p=0.002) — correctly detecting the spatial lag used in the DGP

- Robust LM-Lag remains significant (p=0.039) while Robust LM-Error is not (p=0.66) — pointing to SAR over SEM

- Joint LM-SDM and Joint LM-SLX-Error are both significant — suggesting spatial structure beyond OLS

- LM-WX from SAR null is not significant (p=0.17) — the DGP had no WX effects, so SAR is sufficient

- Robust LM-Error-SDEM is marginally significant (p=0.036) — slight error signal when WX is present

## Method-based API

Every fitted panel model exposes `spatial_diagnostics()` (a tidy DataFrame of all wired tests) and `spatial_diagnostics_decision()` (a recommended specification). The registries are class-aware, so the right tests are run for each model.


In [ ]:
ols_panel.spatial_diagnostics()

In [ ]:
print("OLS panel recommends:", ols_panel.spatial_diagnostics_decision())
print("SAR panel recommends:", sar_panel.spatial_diagnostics_decision())